# Preprocessing Data and Training CNNs

In [8]:
import pickle
import numpy as np
from scipy.signal import butter, filtfilt, resample
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [9]:
# 32 subjects, 20 trials, 32 channels, 59.9s @ 1000hz
# labels are binary [valence, arousal]
b, a = butter(4, [1/500 ,50/500], btype='bandpass')
raw = {}
data = np.zeros((32,20,32,59900//5),dtype=np.float32)
labelz = np.zeros((32,20,2))
for subject in range(1,33):
    if subject % 5 == 0:
        print('Processing Sample '+str(subject))
    with open('MEEG/sample_'+str(subject)+'.dat', 'rb') as f:
        raw[subject] = pickle.load(f, encoding='latin1')
        eeg = raw[subject]['data']
        labelz[subject-1] = raw[subject]['labels']
        # bandpass filter 1-50 Hz
        eeg = filtfilt(b, a, eeg, axis=-1)
        # downsample to 200 Hz
        eeg = resample(eeg, 59900//5, axis=-1)
        # normalize each channel
        mean = eeg.mean(axis=2, keepdims=True)
        std = eeg.std(axis=2, keepdims=True)
        data[subject-1] = (eeg - mean) / std

Processing Sample 5
Processing Sample 10
Processing Sample 15
Processing Sample 20
Processing Sample 25
Processing Sample 30


In [13]:
# make 4s windows, half overlap
def data_windowz(data,labelz,sub_train,sub_test):
    window = 800
    train = []
    trainY = []
    test = []
    testY = []
    # save last two trials of each training subject for testing
    inSampleTest = []
    inSampleTestY = []

    windowz = [train, test, inSampleTest]
    window_labelz = [trainY, testY, inSampleTestY]
    
    for i in range(2):
        for subject in [sub_train,sub_test][i]:
            for trial in range(20):
                for time in range(0, 59900//5 - window + 1, window//2):
                    if i == 0 and trial > 17:
                        windowz[-1].append(data[subject, trial, :, time:time+window])
                        window_labelz[-1].append(labelz[subject, trial])
                    else:  
                        windowz[i].append(data[subject, trial, :, time:time+window])
                        window_labelz[i].append(labelz[subject, trial])

    return windowz, window_labelz

x30, y30 = data_windowz(data,labelz,list(range(30)),list(range(30,32)))
x20, y20 = data_windowz(data,labelz,list(range(20)),list(range(20,32)))
x10, y10 = data_windowz(data,labelz,list(range(10)),list(range(10,32)))

np.savez_compressed('MEEG_windowz.npz',
    x30_train=np.array(x30[0], dtype=np.float32),
    y30_train=np.array(y30[0], dtype=np.float32),
    x30_test=np.array(x30[1], dtype=np.float32),
    y30_test=np.array(y30[1], dtype=np.float32),
    x30_inSample=np.array(x30[2], dtype=np.float32),
    y30_inSample=np.array(y30[2], dtype=np.float32),

    x20_train=np.array(x20[0], dtype=np.float32),
    y20_train=np.array(y20[0], dtype=np.float32),
    x20_test=np.array(x20[1], dtype=np.float32),
    y20_test=np.array(y20[1], dtype=np.float32),
    x20_inSample=np.array(x20[2], dtype=np.float32),
    y20_inSample=np.array(y20[2], dtype=np.float32),

    x10_train=np.array(x10[0], dtype=np.float32),
    y10_train=np.array(y10[0], dtype=np.float32),
    x10_test=np.array(x10[1], dtype=np.float32),
    y10_test=np.array(y10[1], dtype=np.float32),
    x10_inSample=np.array(x10[2], dtype=np.float32),
    y10_inSample=np.array(y10[2], dtype=np.float32))

In [18]:
def build_model(krnl, drop):
    model = nn.Sequential(
    nn.Conv1d(32, 64, kernel_size=krnl, padding=krnl//2),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.MaxPool1d(2),

    nn.Conv1d(64, 128, kernel_size=krnl, padding=krnl//2),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.MaxPool1d(2),

    nn.Conv1d(128, 256, kernel_size=krnl, padding=krnl//2),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.AdaptiveAvgPool1d(1),
    
    nn.Flatten(),
    nn.Linear(256, 64),
    nn.ReLU(),
    nn.Dropout(drop),
    nn.Linear(64, 2))
    return model

def train_model(model, epochz, batch, xData, yData, name):
    X = torch.tensor(np.array(xData), dtype=torch.float32)
    Y = torch.tensor(np.array(yData), dtype=torch.float32)
    train = TensorDataset(X, Y)
    loader = DataLoader(train, batch_size=batch, shuffle=True)
    
    lossF = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(model.parameters(), weight_decay=.0001)
    print('\n Training ' + name + '\n------------------------------')
    for epoch in range(epochz):
        model.train()
        loss_tot = 0
        samplez = 0
        
        for x, y in loader:
            pred = model(x)
            loss = lossF(pred, y)
            loss_tot += loss.item() * x.size(0)
            samplez += x.size(0)

            opt.zero_grad()
            loss.backward()
            opt.step()
        
        if (epoch+1) % 5 == 0:
            print(f'Epoch {epoch+1} Loss = {loss_tot/samplez:.3f}')

    torch.save(model.state_dict(), name + '.pth')


In [22]:
model30 = build_model(7, .25)
train_model(model30, 30,  64, x30[0], y30[0], '30_sample_model')

model20 = build_model(7, .25)
train_model(model20, 30,  64, x20[0], y20[0], '20_sample_model')

model10 = build_model(7, .25)
train_model(model10, 30,  64, x10[0], y10[0], '10_sample_model')


Training 30_sample_model
------------------------------
Epoch 5 Loss = 0.287
Epoch 10 Loss = 0.185
Epoch 15 Loss = 0.128
Epoch 20 Loss = 0.082
Epoch 25 Loss = 0.069
Epoch 30 Loss = 0.056
Training 20_sample_model
------------------------------
Epoch 5 Loss = 0.281
Epoch 10 Loss = 0.172
Epoch 15 Loss = 0.099
Epoch 20 Loss = 0.079
Epoch 25 Loss = 0.059
Epoch 30 Loss = 0.043
Training 10_sample_model
------------------------------
Epoch 5 Loss = 0.220
Epoch 10 Loss = 0.115
Epoch 15 Loss = 0.076
Epoch 20 Loss = 0.072
Epoch 25 Loss = 0.049
Epoch 30 Loss = 0.026
